In [1]:
# Импортиране на библиотеките
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Entities

# Зареждане на набора от данни
path = "data/Entities"
dataset = Entities(path, "AIFB")

C:\Users\PC\anaconda3\envs\gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Processing...
Done!


In [3]:
# Дефиниране на графа
data = dataset[0]
print(data)
num_relations = dataset.num_relations
print(dataset.num_relations)


Data(edge_index=[2, 58086], edge_type=[58086], train_idx=[140], train_y=[140], test_idx=[36], test_y=[36], num_nodes=8285)
90


In [7]:
# Построяване на модел чрез слоя RGCNConv
import torch.nn as nn
from torch_geometric.nn import RGCNConv
class RGCN(nn.Module):

    def __init__(self,
                 num_nodes,
                 hidden_channels,
                 num_classes,
                 num_relations):

        super().__init__()

        self.embedding = nn.Embedding(
            num_nodes,
            hidden_channels
        )

        self.conv1 = RGCNConv(
            hidden_channels,
            hidden_channels,
            num_relations
        )

        self.conv2 = RGCNConv(
            hidden_channels,
            num_classes,
            num_relations
        )

    def forward(self,
                edge_index,
                edge_type):

        x = self.embedding.weight

        x = self.conv1(
            x,
            edge_index,
            edge_type
        )

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index,
            edge_type
        )

        return x

In [8]:
# Създаване на модела
model = RGCN(
    num_nodes=data.num_nodes,
    hidden_channels=16,
    num_classes=dataset.num_classes,
    num_relations=dataset.num_relations
)

In [10]:
# Дефиниране на оптимизатора
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4)

In [12]:
# Обучение на хетерогенната графова невронна мрежа
num_epochs = 100

for epoch in range(num_epochs):
    model.train()

    optimizer.zero_grad()

    out = model(
        data.edge_index,
        data.edge_type
    )

    loss = F.cross_entropy(
        out[data.train_idx],
        data.train_y
    )

    loss.backward()

    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(
            f"Epoch {epoch+1:3d}, "
            f"Loss = {loss.item():.4f}"
        )

Epoch  10, Loss = 0.0359
Epoch  20, Loss = 0.0059
Epoch  30, Loss = 0.0015
Epoch  40, Loss = 0.0015
Epoch  50, Loss = 0.0014
Epoch  60, Loss = 0.0016
Epoch  70, Loss = 0.0019
Epoch  80, Loss = 0.0023
Epoch  90, Loss = 0.0026
Epoch 100, Loss = 0.0029


In [13]:
# Оценяване на модела чрез метриките Accuracy и Macro F1-score
model.eval()

with torch.no_grad():

    out = model(
        data.edge_index,
        data.edge_type
    )

pred = out.argmax(dim=1)

In [14]:
# Изчисляване на мерките
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
accuracy = accuracy_score(
    data.test_y.cpu(),
    pred[data.test_idx].cpu()
)

macro_f1 = f1_score(
    data.test_y.cpu(),
    pred[data.test_idx].cpu(),
    average="macro"
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

Accuracy : 0.9444
Macro F1 : 0.9003
